In [20]:
from pathlib import Path
import json

In [21]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260402"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=3
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [22]:
CONFIG_PATH = Path(f"{TARGET_DIR}/{TARGET_DATE}/{TARGET_TYPE}/{TARGET_CONFIG}")

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

best = cfg["best_encoding"]
HUFFMAN_CODES = best["huffman_codes"]  # symbol(str) → bitstring
MERGED_VALUES = set(best["merged_values"])
FALLBACK_SYMBOL = best["representative_value"] # special rare-delta token
FALLBACK_CODE = HUFFMAN_CODES[str(FALLBACK_SYMBOL)]
BITSLENGTH = best["bitsLen_symNum"]
MIN_BITS_LENGTH = best["min_code_length"]
MAX_BITS_LENGTH = best["max_code_length"]

# Build reverse map  bitstring → int symbol  (for decoding)
REVERSE_CODES = {v: int(k) for k, v in HUFFMAN_CODES.items()}

print(f"[config]  total symbols in Huffman table : {len(HUFFMAN_CODES)}")
print(f"[config]  merged (rare) values           : {len(MERGED_VALUES)}")
print(f"[config]  fallback code for 1000         : {FALLBACK_CODE}")
avg_bits = cfg["best_avg_bits"]
print(
    f"[config]  best avg bits/symbol           : {avg_bits:.4f}  "
    f"(vs 8 raw bits → {avg_bits/8*100:.1f}% of original)"
)

[config]  total symbols in Huffman table : 262
[config]  merged (rare) values           : 250
[config]  fallback code for 1000         : 000100
[config]  best avg bits/symbol           : 7.0287  (vs 8 raw bits → 87.9% of original)


In [23]:
header_var = cfg["header_package"]
V_T = header_var["V_T"]
U_T = header_var["U_T"]
current_ver = header_var["current_ver"]
update_ver = header_var["update_ver"]
nb_symbols = header_var["nb_symbols"]
min_code_length = header_var["min_code_length"]
max_code_length = header_var["max_code_length"]
RWID = header_var["rwid"]
SWID = header_var["swid"]
target_size = header_var["target_size"]
payload_bits_len = header_var["payload_bits_len"]
target_checksum = header_var["target_checksum"]

md5_bytes = bytes.fromhex(target_checksum)

In [24]:
# Split into 2 frames (8 bytes each)
frame1 = md5_bytes[:8]
frame2 = md5_bytes[8:]

# Convert to little endian (reverse each frame)
frame1_le = frame1[::-1]
frame2_le = frame2[::-1]

print("Frame 1 (LE):", frame1_le.hex())
print("Frame 2 (LE):", frame2_le.hex())

Frame 1 (LE): ee09049fd68fa27d
Frame 2 (LE): 725803f4808425ca


In [25]:
len(HUFFMAN_CODES)

262

In [26]:
import struct
import zlib
from dataclasses import dataclass

MAGIC = b"Q-HUFDEL"  # exactly 8 bytes — frame 1


# Data class representing the decoded header
@dataclass
class HufDelHeader:
    # Frame 2
    version_type: int  # V-T  (1 byte)
    update_type: int  # U-T  (1 byte)
    version_current: int  # current version of dest file  (2 bytes, LE)
    version_update: int  # update version of dest file   (2 bytes, LE)
    num_huffman_symbols: int  # number of Huffman symbols (2 byte LE)

    # Frame 3
    min_code_len: int  # min code length of symbol      (1 byte)
    max_code_len: int  # max code length of symbol      (1 byte)
    replaced_bits: int  # replaced value bit-width, e.g. 8 (1 byte)
    symbol_bits: int  # symbol value bit-width (1 byte)
    special_symbol: int  # symbol with special handling (e.g. merged values) (2 byte)
    # ↑ 2 bytes used — 2 bytes reserved padding to fill frame 3

    # Frame 4: checksum placeholder (0xFF * 8) — algorithm MD5
    checksum: bytes  # 16-byte checksum (MD5) stored as two 8-byte LE frames

    # Frame 5
    target_size_bytes: int  # size of model2 in bytes        (4 bytes, LE)
    bitstream_size_bits: int  # total bitstream payload bytes  (4 bytes, LE)


# Encoder
HEADER_FRAME_COUNT = 6
HEADER_SIZE = HEADER_FRAME_COUNT * 8  # 48 bytes


def encode_header(h: HufDelHeader) -> bytes:
    """
    Encode the 5-frame (40-byte) package header.

    Frame layout:
        [0]  Magic          "Q-HUFDEL"                          8 bytes
        [1]  V-T | U-T | ver_dest (LE,2) | ver_upd (LE,2)
             | n_symbols                                        8 bytes
        [2]  min_code | max_code | replaced_bits | symbol_bits
             | _pad (4)                                         8 bytes
        [3]  Checksum placeholder 0xFF*8                        8 bytes
        [4]  model2_size (LE,4) | bitstream_size (LE,4)         8 bytes
    """

    # ── Frame 1: magic ────────────────────────────────────────────────────────
    frame1 = MAGIC  # 8 bytes

    # ── Frame 2: version / type flags ────────────────────────────────────────
    frame2 = struct.pack(
        "<BBHHH",  # little-endian
        h.version_type,  # 1 byte  — V-T
        h.update_type,  # 1 byte  — U-T
        h.version_current,  # 2 bytes — current current version
        h.version_update,  # 2 bytes — update current version
        h.num_huffman_symbols,  # 2 byte  — symbol count
    )  # = 8 bytes

    # ── Frame 3: code / bit-width metadata ───────────────────────────────────
    frame3 = struct.pack(
        "<BBBBh2x",  # 4 bytes data + 4 bytes padding
        h.min_code_len,  # 1 byte
        h.max_code_len,  # 1 byte
        h.replaced_bits,  # 1 byte
        h.symbol_bits,  # 1 byte
        h.special_symbol,  # 2 bytes
    )  # = 8 bytes

    # ── Frame 4: checksum placeholder ──────────────────────────────────
    f1_int = int.from_bytes(h.checksum[:8], byteorder="big")
    f2_int = int.from_bytes(h.checksum[8:], byteorder="big")
    frame4 = struct.pack("<QQ", f1_int, f2_int)

    # ── Frame 5: sizes ────────────────────────────────────────────────────────
    frame5 = struct.pack(
        "<II",  # two unsigned 32-bit LE integers
        h.target_size_bytes,  # 4 bytes — model2 file size
        h.bitstream_size_bits,  # 4 bytes — bitstream payload size
    )  # = 8 bytes

    header = frame1 + frame2 + frame3 + frame4 + frame5
    assert len(header) == HEADER_SIZE, f"Header must be {HEADER_SIZE} bytes"
    return header

In [27]:
header = HufDelHeader(
    version_type=V_T,
    update_type=U_T,
    version_current=current_ver,  # e.g. v1.0
    version_update=update_ver,  # e.g. v1.1
    num_huffman_symbols=nb_symbols,
    min_code_len=min_code_length,
    max_code_len=max_code_length,
    replaced_bits=RWID,
    symbol_bits=SWID,
    special_symbol=FALLBACK_SYMBOL,
    checksum=md5_bytes,
    target_size_bytes=target_size,
    bitstream_size_bits=payload_bits_len,
)

encoded_header = encode_header(header)

In [28]:
for i in range(0, len(encoded_header), 8):
    chunk = encoded_header[i:i+8]
    print(' '.join(f'{b:02X}' for b in chunk))

51 2D 48 55 46 44 45 4C
00 00 01 00 03 00 06 01
04 0C 08 09 3E FF 00 00
EE 09 04 9F D6 8F A2 7D
72 58 03 F4 80 84 25 CA
80 21 04 00 60 D9 1C 00


In [29]:
with open(f"{TARGET_DIR}/package_header_{MODEL_ID1}_{MODEL_ID2}.bin", "wb") as f:
    f.write(encoded_header)

In [30]:
# Decoder
def decode_header(data: bytes) -> tuple[HufDelHeader, bytes]:
    """
    Parse the 40-byte header from the front of a package.

    Returns:
        (HufDelHeader, remainder)  — remainder is everything after the header.

    Raises:
        ValueError  on magic mismatch or truncated data.
    """
    if len(data) < HEADER_SIZE:
        raise ValueError(
            f"Package too short: expected at least {HEADER_SIZE} bytes, "
            f"got {len(data)}."
        )

    # ── Frame 1: magic check ──────────────────────────────────────────────────
    magic = data[0:8]
    if magic != MAGIC:
        raise ValueError(f"Magic mismatch: expected {MAGIC!r}, got {magic!r}")

    # ── Frame 2: version / type flags ────────────────────────────────────────
    (
        version_type,
        update_type,
        version_current,
        version_update,
        num_huffman_symbols,
    ) = struct.unpack_from("<BBHHH", data, offset=8)

    # ── Frame 3: code / bit-width metadata ───────────────────────────────────
    (
        min_code_len,
        max_code_len,
        replaced_bits,
        symbol_bits,
        special_symbol,
    ) = struct.unpack_from("<BBBBh", data, offset=16)

    # ── Frame 4: checksum placeholder
    (f1_le, f2_le) = struct.unpack_from("<QQ", data, offset=24)

    # Convert to big endian (reverse bytes)
    f1_be = struct.pack(">Q", f1_le)
    f2_be = struct.pack(">Q", f2_le)
    checksum = f1_be + f2_be

    # ── Frame 5: sizes ────────────────────────────────────────────────────────
    (
        target_size_bytes,
        bitstream_size_bits,
    ) = struct.unpack_from("<II", data, offset=40)

    header = HufDelHeader(
        version_type=version_type,
        update_type=update_type,
        version_current=version_current,
        version_update=version_update,
        num_huffman_symbols=num_huffman_symbols,
        min_code_len=min_code_len,
        max_code_len=max_code_len,
        replaced_bits=replaced_bits,
        symbol_bits=symbol_bits,
        special_symbol=special_symbol,
        checksum=checksum,
        target_size_bytes=target_size_bytes,
        bitstream_size_bits=bitstream_size_bits,
    )

    remainder = data[HEADER_SIZE:]
    return header, remainder

In [31]:
header

HufDelHeader(version_type=0, update_type=0, version_current=1, version_update=3, num_huffman_symbols=262, min_code_len=4, max_code_len=12, replaced_bits=8, symbol_bits=9, special_symbol=-194, checksum=b'}\xa2\x8f\xd6\x9f\x04\t\xee\xca%\x84\x80\xf4\x03Xr', target_size_bytes=270720, bitstream_size_bits=1890656)